# 🧠 AN·RA Master Training Notebook

**Full-stack autonomous training pipeline for Google Colab.**

This notebook trains An-Ra from scratch using Colab's free T4 GPU. Every subsystem is activated:
- **TurboQuant** — 4-bit KV-cache compression (6× memory savings)
- **Ouroboros** — Recursive self-refinement during training
- **Symbolic Bridge** — Math/logic/code verification of training data
- **Identity Conditioning** — Curriculum-based identity fine-tuning
- **Sovereignty Audit** — Post-training integrity & rollback checks
- **Ghost Memory** — Persistent semantic memory population

No API keys required. No external dependencies beyond PyTorch.

In [ ]:
#@title 🔧 Cell 1: Environment Setup & GPU Check
import torch, subprocess, sys, os

print('=' * 50)
print('AN·RA TRAINING ENVIRONMENT')
print('=' * 50)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')
    raise RuntimeError('GPU required for training')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/AnRa'
os.makedirs(DRIVE, exist_ok=True)
print(f'✅ Drive mounted: {DRIVE}')

In [ ]:
#@title 📦 Cell 2: Clone Repo & Install Dependencies
import os, subprocess, sys
REPO = '/content/An-Ra'

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)

os.chdir(REPO)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

# Initialize path system
sys.path.insert(0, REPO)
from anra_paths import inject_all_paths, ensure_dirs, ROOT
inject_all_paths()
ensure_dirs()

print(f'\n✅ Repository ready at {REPO}')
print(f'   Root: {ROOT}')


In [ ]:
#@title 🏥 Cell 3: System Health Check — Verify All Subsystems
from anra_paths import (
    CORE_DIR, TRAINING_DIR, INFERENCE_DIR, OUROBOROS_DIR,
    SYMBOLIC_BRIDGE_DIR, SOVEREIGNTY_DIR, IDENTITY_DIR,
    GHOST_MEMORY_DIR, MEMORY_DIR, AGENT_LOOP_DIR,
    TRAINING_DATA_DIR, TOKENIZER_DIR, get_tokenizer_file,
    get_dataset_file, get_optimization_config
)
import importlib

def _try_import(module_name, attr=None):
    try:
        m = importlib.import_module(module_name)
        if attr:
            return getattr(m, attr)
        return m
    except Exception as e:
        raise ImportError(f'{module_name}: {e}')

checks = {
    'anra_brain (CausalTransformer)': lambda: _try_import('anra_brain', 'CausalTransformer'),
    'generate (inference engine)': lambda: _try_import('generate', 'generate'),
    'TurboQuant (KV-cache)': lambda: _try_import('core.turboquant', 'CompressedKVCache'),
    'Ouroboros (recursive reasoning)': lambda: _try_import('ouroboros_numpy', 'OuroborosNumpy'),
    'Symbolic Bridge (math/logic)': lambda: _try_import('symbolic_bridge'),
    'Identity Injector': lambda: _try_import('identity_injector', 'get_identity_injector'),
    'Sovereignty Bridge': lambda: _try_import('sovereignty_bridge'),
    'Optimizations (scheduler/detector)': lambda: _try_import('optimizations', 'AdaptiveScheduler'),
    'Curriculum (training phases)': lambda: _try_import('training.curriculum', 'get_phase'),
    'Tokenizer': lambda: get_tokenizer_file().exists(),
    'Training Dataset': lambda: get_dataset_file().exists(),
    'Optimization Config': lambda: get_optimization_config().exists(),
}

ok_count = 0
for name, check in checks.items():
    try:
        check()
        print(f'  ✅ {name}')
        ok_count += 1
    except Exception as e:
        print(f'  ❌ {name}: {e}')

print(f'\n{ok_count}/{len(checks)} subsystems operational')
if ok_count == len(checks):
    print('🟢 ALL SYSTEMS GO — ready for training')
else:
    print('🟡 Some subsystems unavailable — training will proceed with available modules')


In [ ]:
#@title 🏗️ Cell 4: Build Base Brain (skip if checkpoint exists)
import os, shutil, subprocess, sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/AnRa'
brain_exists = any(Path(p).exists() for p in [
    f'{DRIVE}/anra_brain.pt',
    'anra_brain.pt',
    f'{DRIVE}/anra_brain_identity.pt',
    'anra_brain_identity.pt'
])

if brain_exists:
    print('✅ Checkpoint found — skipping base brain training')
    for name in ['anra_brain_identity.pt', 'anra_brain.pt']:
        drive_ckpt = Path(f'{DRIVE}/{name}')
        local_ckpt = Path(name)
        if drive_ckpt.exists() and not local_ckpt.exists():
            shutil.copy2(drive_ckpt, local_ckpt)
            print(f'  Copied {name} from Drive ({drive_ckpt.stat().st_size/1e6:.1f} MB)')
else:
    print('🔨 No checkpoint found — building base brain from scratch...')
    subprocess.check_call([sys.executable, 'scripts/build_brain.py',
        '--data', 'training_data/anra_dataset_v6_1.txt',
        '--epochs', '20', '--batch_size', '64'])
    if Path('anra_brain.pt').exists():
        shutil.copy2('anra_brain.pt', f'{DRIVE}/anra_brain.pt')
        print('✅ Base brain saved to Drive')


In [ ]:
#@title 🎯 Cell 5: Identity Fine-Tuning (Full Pipeline)
import subprocess, sys, shutil
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'training.finetune_anra'])

# Copy checkpoint to Drive
ckpt = Path('anra_brain_identity.pt')
if ckpt.exists():
    shutil.copy2(ckpt, f'/content/drive/MyDrive/AnRa/{ckpt.name}')
    print(f'\n✅ Identity checkpoint saved to Drive ({ckpt.stat().st_size/1e6:.1f} MB)')


In [ ]:
#@title 🧬 Cell 6: Ouroboros Recursive Training (Optional — Advanced)
#@markdown Trains the Ouroboros recursive reasoning layer.

ENABLE_OUROBOROS = True  #@param {type:"boolean"}

import subprocess, sys
from pathlib import Path

if ENABLE_OUROBOROS:
    trainer = Path('scripts/train_ouroboros.py')
    subprocess.check_call([sys.executable, str(trainer),
        '--base_model', 'anra_brain_identity.pt', '--steps', '5000'])
    print('✅ Ouroboros training complete')
else:
    print('⏭️  Ouroboros training skipped')


In [ ]:
#@title 🧠 Cell 7: Populate Memory System
import subprocess, sys
subprocess.check_call([sys.executable, 'scripts/populate_memory.py'])
print('✅ Memory system populated and saved to Drive')


In [ ]:
#@title 📊 Cell 8: Self-Improvement Analysis
import subprocess, sys
subprocess.check_call([sys.executable, 'scripts/run_self_improvement.py'])
print('\n✅ Self-improvement report saved to Drive')


In [ ]:
#@title 🛡️ Cell 9: Sovereignty Audit (Post-Training Integrity Check)
import subprocess, sys
subprocess.check_call([sys.executable, 'scripts/run_sovereignty_audit.py'])
print('\n✅ Sovereignty audit complete')


In [ ]:
#@title 🧪 Cell 10: Test Suite (24 tests)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'tests.test_suite'],
    cwd='/content/An-Ra')


In [ ]:
#@title 💬 Cell 11: Interactive Chat (Test Your Model)
from generate import generate

prompts = [
    'H: Who are you?\nANRA:',
    'H: What is your purpose?\nANRA:',
    'H: Explain quantum computing in simple terms.\nANRA:',
    'H: Write a Python function to check if a number is prime.\nANRA:',
]

print('=' * 60)
print('AN·RA IDENTITY VERIFICATION')
print('=' * 60)
for prompt in prompts:
    response = generate(prompt, strategy='nucleus', max_tokens=150,
                       use_identity=True, use_ouroboros=True,
                       use_symbolic=True, use_turboquant=True)
    q = prompt.split('H: ')[1].split('\n')[0]
    print(f'\n🧑 {q}')
    print(f'🤖 {response[:300]}')
    print('-' * 60)

In [ ]:
#@title 🚀 Cell 12: Launch API Server (Optional)
#@markdown Starts the FastAPI server with ngrok tunnel for remote access.

LAUNCH_API = False  #@param {type:"boolean"}

if LAUNCH_API:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok
    import threading, uvicorn
    from app import app

    def run_server():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()

    public_url = ngrok.connect(8000)
    print(f"\n🌐 AN·RA API is live at: {public_url}")
    print(f"   Health: {public_url}/health")
    print(f"   Docs:   {public_url}/docs")
else:
    print("API server not launched. Set LAUNCH_API = True to start.")
